In [1]:
# Imports
from NavSysLib.utilities import *
from NavSysLib.Coords import *
from NavSysLib.Orbit import *

from pathlib import Path

Exercise 1

In [2]:
sat_orbit = Orbit.from_orbital_parameters(a=26559755, e=0.017545, arg_perigee=1.626021)
t_passed = 39929
print(sat_orbit.parameters_at_time(t_passed))

{'T': np.float64(43077.158247828884), 'eta': np.float64(0.00014585886262579226), 'M': np.float64(5.823998525785259), 'E': np.float64(5.816098235222907), 'r': np.float64(26143679.306235045), 'true_anomaly': np.float64(-0.47505030825905525), 'arg_latitude': np.float64(1.1509706917409446)}


Exercise 2

In [3]:
gps_orbit = Orbit.from_orbital_parameters(a=5154**2, e=0.0076, toe=540000, M0=-0.820, delta_n=0, wn=2056, mu=3.986005e14)
print(gps_orbit.first_perigee_tow_in_week())

28581.589445878257


Exercise 3

In [4]:
delta_t = gps_delta_t(2057, 600830, 2058, 86)
print(delta_t)

-4056.0


Exercise 4

In [5]:
ephemeris = Ephemeris(
    wn=2056, 
    t_oe=532800, 
    sqrt_a=5.154e3, 
    d_eta=4.869e-9, 
    M_0=-2.185e-1, 
    e=1.889e-2, 
    omega=-1.737, 
    i_0=9.552e-1,
    idot=4.464e-11,
    Omega_0=3.108,
    Omega_dot=-8.740e-9,
    C_uc=0.0,
    C_us=0.0,
    C_rc=0.0,
    C_rs=0.0,
    C_ic=0.0,
    C_is=0.0,
)

orbit = Orbit(ephemeris=ephemeris)

sat_coords = WGS84Coords.from_orbit(orbit, wn=2056, tow=536400)
print(sat_coords.to_ecef_string())

x=14302152.43 m, y=5722354.63 m, z=-21052146.25 m


Exercise 5

In [6]:
r1 = WGS84Coords.from_ecef(4918525.18, -791212.21, 3969762.19)

az, el = r1.az_el_to(sat_coords)
print(f"Azimuth: {az:.2f} degrees, Elevation: {el:.2f} degrees")

e: 7921217.32550902, n: -24667559.001334067, u: -9238006.523915734
Azimuth: 162.20 degrees, Elevation: -19.62 degrees


Not in line of sight (negative elevation)

Exercise 6

In [7]:
eph_path = Path("ub1.ubx.2056.540000a.eph")
if not eph_path.exists():
    eph_path = Path("Exercises/E4/ub1.ubx.2056.540000a.eph")

wn = 2056
tow = 536400

with eph_path.open("r", encoding="ascii") as f:
    eph_lines = [line.strip() for line in f if line.strip()]

orbits = [Orbit.from_eph_line(line, reference_wn=wn) for line in eph_lines]

print(f"Loaded {len(eph_lines)} lines from {eph_path}")
print(f"Parsed {len(orbits)} orbits")

# Print direct ECEF from orbit propagation to avoid tiny round-trip conversion drift.
for orbit in orbits:
    x, y, z = orbit.wgs84_ecef_position(wn=wn, tow=tow)
    print(f"SV{orbit.ephemeris.sv_num:02d}: WN={orbit.ephemeris.wn}, x={x:.2f} m, y={y:.2f} m, z={z:.2f} m")

Loaded 15 lines from ub1.ubx.2056.540000a.eph
Parsed 15 orbits
SV02: WN=2056, x=14300556.71 m, y=5726823.97 m, z=-21048150.88 m
SV05: WN=2056, x=24017954.37 m, y=-254576.22 m, z=-11683289.16 m
SV09: WN=2056, x=-337427.23 m, y=17906386.15 m, z=-19648806.74 m
SV10: WN=2056, x=-5844820.64 m, y=-14047605.20 m, z=21837695.43 m
SV12: WN=2056, x=23594489.43 m, y=-10613395.40 m, z=-5810709.92 m
SV13: WN=2056, x=20975774.71 m, y=9577789.64 m, z=13114921.96 m
SV15: WN=2056, x=19235496.08 m, y=-2940584.75 m, z=17976624.26 m
SV17: WN=2056, x=13432672.93 m, y=21227658.05 m, z=9167271.45 m
SV19: WN=2056, x=17813675.55 m, y=19604058.39 m, z=1008273.56 m
SV20: WN=2056, x=3923216.69 m, y=-17848331.09 m, z=19121558.71 m
SV21: WN=2056, x=877752.50 m, y=-26500879.48 m, z=3407085.71 m
SV24: WN=2056, x=14306205.39 m, y=-14437110.53 m, z=16769402.10 m
SV25: WN=2056, x=13797647.62 m, y=-16317824.62 m, z=-15822957.82 m
SV28: WN=2056, x=4007666.61 m, y=14488235.89 m, z=22502789.83 m
SV30: WN=2056, x=266018.22 m

Exercise 7

In [11]:
r1_xyz = (4918525.18, -791212.21, 3969762.19)
wn = 2056
tow = 536400
epsilon = 1e-3 # 1 mm

sat_positions = []

for orbit in orbits:
    t_tx = orbit.get_tx_time_from_ref_point(wn, tow, r1_xyz, epsilon=epsilon)
    pos = orbit.get_pos_at_tx_time(t_tx, ref_wn=wn, ref_tow=tow)
    sat_positions.append(pos)
    print(f"SV{orbit.ephemeris.sv_num:02d}: WN={orbit.ephemeris.wn}, t_tx={t_tx:.9f} s, x={pos[0]:.2f} m, y={pos[1]:.2f} m, z={pos[2]:.2f} m")

Iteration 1: t_tx=1244005200.000000 s, s = [ 14300556.70954257   5726823.97414121 -21048150.87560661] m, d_Omega=0.000000000e+00 rad, d=0.000 m, d_prime=27502786.867 m
Iteration 2: t_tx=1244005199.908261 s, s = [ 14300630.18284382   5726472.32012137 -21048194.02200781] m, d_Omega=6.689744299e-06 rad, d=27502786.867 m, d_prime=27502767.842 m
Iteration 3: t_tx=1244005199.908261 s, s = [ 14300630.18281732   5726472.32018755 -21048194.02200781] m, d_Omega=6.689739672e-06 rad, d=27502767.842 m, d_prime=27502767.842 m
Transmission time calculation converged in 3 iterations with final distance 27502767.842 m
SV02: WN=2056, t_tx=1244005199.908260584 s, x=14300630.18 m, y=5726472.32 m, z=-21048194.02 m
Iteration 1: t_tx=1244005200.000000 s, s = [ 24017954.36753457   -254576.21880484 -11683289.16180649] m, d_Omega=0.000000000e+00 rad, d=0.000 m, d_prime=24700084.820 m
Iteration 2: t_tx=1244005199.917609 s, s = [ 24018057.80734222   -254782.68302484 -11683071.42664094] m, d_Omega=6.008018475e-06 

Exercise 8

In [13]:
# Get direction cosines in ECEF and ENU from r1 to each satellite
r1_coords = WGS84Coords.from_ecef(*r1_xyz)
for position in sat_positions:
    sat_coords = WGS84Coords.from_ecef(*position)
    dir_cos_ecef = r1_coords.direction_cosines_to(sat_coords)
    dir_cos_enu = r1_coords.direction_cosines_to_enu(sat_coords)
    print(f"Direction cosines to satellite at x={position[0]:.2f} m, y={position[1]:.2f} m, z={position[2]:.2f} m:")
    print(f"  ECEF: {dir_cos_ecef}")
    print(f"  ENU: {dir_cos_enu}")

Direction cosines to satellite at x=14300630.18 m, y=5726472.32 m, z=-21048194.02 m:
  ECEF: (np.float64(0.3411331165975851), np.float64(0.23698285577586073), np.float64(-0.9096523087579869))
  ENU: (np.float64(0.2881543033197185), np.float64(-0.89675070592342), np.float64(-0.33586495635028013))
Direction cosines to satellite at x=24018057.81 m, y=-254782.68 m, z=-11683071.43 m:
  ECEF: (np.float64(0.7732597286915732), np.float64(0.0217177749395347), np.float64(-0.6337173898793611))
  ENU: (np.float64(0.14425270145933558), np.float64(-0.9698831565123996), np.float64(0.1962595751427888))
Direction cosines to satellite at x=-337090.27 m, y=17906540.79 m, z=-19648671.23 m:
  ECEF: (np.float64(-0.1718716849947655), np.float64(0.6114629859641517), np.float64(-0.7723814735562109))
  ENU: (np.float64(0.5764048490786754), np.float64(-0.43551875510750077), np.float64(-0.6914339186850775))
Direction cosines to satellite at x=-5845119.18 m, y=-14047493.88 m, z=21837688.71 m:
  ECEF: (np.float64(-

Exercise 9

In [14]:
for position in sat_positions:
    sat_coords = WGS84Coords.from_ecef(*position)
    az, el = r1_coords.az_el_to(sat_coords)
    print(f"Azimuth: {az:.2f} degrees, Elevation: {el:.2f} degrees")

e: 7925040.986141884, n: -24663126.723842513, u: -9237216.013156034
Azimuth: 162.19 degrees, Elevation: -19.63 degrees
e: 3563044.958676204, n: -23956135.70045086, u: 4847615.904105427
Azimuth: 171.54 degrees, Elevation: 11.32 degrees
e: 17625720.185095374, n: -13317604.328200433, u: -21143161.437151127
Azimuth: 127.07 degrees, Elevation: -43.74 degrees
e: -14797525.496141726, n: 19269789.94220904, u: 4533904.199209653
Azimuth: -37.52 degrees, Elevation: 10.57 degrees
e: -6731512.1177494405, n: -20143498.937241044, u: 9479121.229575586
Azimuth: -161.52 degrees, Elevation: 24.05 degrees
e: 12787427.417989364, n: -1756325.963182224, u: 16804190.613546137
Azimuth: 97.82 degrees, Elevation: 52.47 degrees
e: 151543.66393706948, n: 1866958.3772337586, u: 20056898.03974601
Azimuth: 4.64 degrees, Elevation: 84.66 degrees
e: 23091627.451113176, n: 982016.2978792903, u: 7081493.248612298
Azimuth: 87.56 degrees, Elevation: 17.03 degrees
e: 22184346.080105823, n: -8250149.276229431, u: 5550937.655